In [0]:
import requests as rq
import json as js
from datetime import datetime

In [0]:
BASE_URL = "https://api.openbrewerydb.org/v1/breweries"
PER_PAGE = 200

In [0]:
def get_breweries():
    all_data = []
    page = 1
    while True:
        try:
            response = rq.get(
                BASE_URL,
                params={"page": page, "per_page": PER_PAGE},
            )
            response.raise_for_status()
            batch = response.json()
            if not batch:
                print(f"Coleta Concluída. Total de páginas: {page -1}")
                break
            all_data.extend(batch)
            print(f"Página {page} - {len(batch)} registros coletados")
            page += 1
        except rq.exceptions.RequestException as error:
            print(f"Erro na página {page}: {error}")
            break
    return all_data
    




In [0]:
breweries = get_breweries()
print(f"Total coletado: {len(breweries)}") 
print(f"Primeiro item: {breweries[0]} e último item: {breweries[-1]}")     

In [0]:
today = datetime.now()
year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d")

volume_path = (f"/Volumes/lakehouse_portfolio/bronze/breweries_raw"
               f"/year={year}/month={month}/day={day}/date.json"
               
               )
text_json = js.dumps(breweries, ensure_ascii=False, indent=2)
dbutils.fs.put(volume_path, text_json, overwrite=True)

folder = f"/Volumes/lakehouse_portfolio/bronze/breweries_raw/year={year}/month={month}/day={day}"
files = dbutils.fs.ls(folder)

print(f"Total de arquivos: {len(files)}")
print(f"Arquivos: {files}")
for f in files:
    kb_lenghth = round(f.size/1024, 1)
    print(f"Arquivo: {f.name} - {kb_lenghth} KB")


In [0]:
import json 
conteudo_lido = dbutils.fs.head(volume_path, 5000)
dados_verificados = json.loads(conteudo_lido[:conteudo_lido.rfind('}')+1] + "]")
for breweries in dados_verificados[:2]:
    print(json.dumps(breweries, indent=2, ensure_ascii=False))
    print("---")